# Popularity baseline report

Minimal report for the completed MovieLens and Steam `popularity_f1_threshold` runs. The notebook reads existing `metrics.json` artifacts from `outputs/in_distribution/interaction_prediction`, keeps the latest run for each `(dataset, negative_ratio, seed)`, and reports test metrics as `mean ± std` across seeds.

## Reproduction commands

These are the commands used to generate the popularity artifacts summarized below:

```bash
mkdir -p outputs/logs

.venv/bin/python - <<'PY' > /tmp/bcs_ml_tasks.txt
from runners.in_distribution.interaction_prediction.task_builders import TASK_BUILDERS
for name in TASK_BUILDERS:
    if name.startswith("ml-1m_"):
        print(name)
PY

.venv/bin/python - <<'PY' > /tmp/bcs_steam_tasks.txt
from runners.in_distribution.interaction_prediction.task_builders import TASK_BUILDERS
for name in TASK_BUILDERS:
    if name.startswith("steam_"):
        print(name)
PY

.venv/bin/python -m runners.in_distribution.interaction_prediction.run \
  --tasks "$(paste -sd, /tmp/bcs_ml_tasks.txt)" \
  --methods popularity_f1_threshold \
  > outputs/logs/ml_popularity.log 2>&1 &

.venv/bin/python -m runners.in_distribution.interaction_prediction.run \
  --tasks "$(paste -sd, /tmp/bcs_steam_tasks.txt)" \
  --methods popularity_f1_threshold \
  > outputs/logs/steam_popularity.log 2>&1 &
```

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and path.name == "beyond-click-sim":
            return path
    raise RuntimeError("Could not find beyond-click-sim repo root")


REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "outputs" / "in_distribution" / "interaction_prediction"
METHOD = "popularity_f1_threshold"
TASK_RE = re.compile(
    r"^(?P<dataset>ml-1m|steam)_interaction_cap20_m(?P<negative_ratio>\d+)_seed(?P<seed>\d+)$"
)

rows = []
for path in sorted(RESULTS_ROOT.glob(f"*_{METHOD}/metrics.json")):
    metrics = json.loads(path.read_text())
    match = TASK_RE.match(metrics["task"])
    if match is None:
        continue

    test_macro = metrics["test"]["macro_by_group"]
    test_micro = metrics["test"]["micro"]
    val_macro = metrics["val"]["macro_by_group"]
    val_micro = metrics["val"]["micro"]
    rows.append(
        {
            "run_dir": path.parent.name,
            "dataset": match.group("dataset"),
            "negative_ratio": int(match.group("negative_ratio")),
            "seed": int(match.group("seed")),
            "test_macro_accuracy": test_macro["accuracy"],
            "test_macro_precision": test_macro["precision"],
            "test_macro_recall": test_macro["recall"],
            "test_macro_f1": test_macro["f1"],
            "test_micro_accuracy": test_micro["accuracy"],
            "test_micro_precision": test_micro["precision"],
            "test_micro_recall": test_micro["recall"],
            "test_micro_f1": test_micro["f1"],
            "val_macro_f1": val_macro["f1"],
            "val_micro_f1": val_micro["f1"],
            "threshold": metrics["threshold"],
            "test_n": test_macro["n"],
            "test_n_groups": test_macro["n_groups"],
        }
    )

runs = pd.DataFrame(rows)
if runs.empty:
    raise RuntimeError(f"No {METHOD} metrics found under {RESULTS_ROOT}")

runs["timestamp"] = runs["run_dir"].str.extract(r"^(\d{8}T\d{6}Z)_")
runs = (
    runs.sort_values(["dataset", "negative_ratio", "seed", "timestamp"])
    .drop_duplicates(["dataset", "negative_ratio", "seed"], keep="last")
    .sort_values(["dataset", "negative_ratio", "seed"])
    .reset_index(drop=True)
)

runs.shape

(50, 18)

In [2]:
coverage = (
    runs.groupby(["dataset", "negative_ratio"])
    .agg(seeds=("seed", "nunique"), runs=("seed", "size"), test_n=("test_n", "mean"), test_groups=("test_n_groups", "mean"))
    .astype({"seeds": int, "runs": int, "test_n": int, "test_groups": int})
)
coverage

seeds  runs   test_n  test_groups
dataset negative_ratio                                   
ml-1m   1                   5     5   110044         6040
        2                   5     5   106989         6040
        3                   5     5   120456         6040
        9                   5     5   120800         6040
        19                  5     5   120800         6040
steam   1                   5     5   922194        57333
        2                   5     5   944436        57333
        3                   5     5  1082604        57333
        9                   5     5  1146660        57333
        19                  5     5  1146660        57333

In [3]:
metric_columns = {
    "macro accuracy": "test_macro_accuracy",
    "macro precision": "test_macro_precision",
    "macro recall": "test_macro_recall",
    "macro F1": "test_macro_f1",
    "micro accuracy": "test_micro_accuracy",
    "micro precision": "test_micro_precision",
    "micro recall": "test_micro_recall",
    "micro F1": "test_micro_f1",
}

summary_parts = []
for label, column in metric_columns.items():
    stats = runs.groupby(["dataset", "negative_ratio"])[column].agg(["mean", "std"])
    summary_parts.append(
        pd.DataFrame(
            {
                label: stats["mean"].map("{:.4f}".format) + " +/- " + stats["std"].map("{:.4f}".format)
            }
        )
    )

test_summary = pd.concat(summary_parts, axis=1)
test_summary

macro accuracy    macro precision  \
dataset negative_ratio                                         
ml-1m   1               0.7686 +/- 0.0034  0.7250 +/- 0.0072   
        2               0.7670 +/- 0.0043  0.6288 +/- 0.0083   
        3               0.7892 +/- 0.0023  0.5773 +/- 0.0046   
        9               0.8564 +/- 0.0050  0.3983 +/- 0.0068   
        19              0.8928 +/- 0.0099  0.2571 +/- 0.0082   
steam   1               0.8898 +/- 0.0006  0.8717 +/- 0.0026   
        2               0.8916 +/- 0.0012  0.8215 +/- 0.0049   
        3               0.8984 +/- 0.0008  0.7817 +/- 0.0029   
        9               0.9329 +/- 0.0004  0.6733 +/- 0.0022   
        19              0.9477 +/- 0.0016  0.5107 +/- 0.0038   

                             macro recall           macro F1  \
dataset negative_ratio                                         
ml-1m   1               0.8923 +/- 0.0091  0.7941 +/- 0.0006   
        2               0.8218 +/- 0.0126  0.7014 +/- 0.0005   
        3               0.7577 +/- 0.0055  0.6408 +/- 0.0007   
        9               0.6143 +/- 0.0151  0.4570 +/- 0.0024   
        19              0.5418 +/- 0.0318  0.3282 +/- 0.0049   
steam   1               0.9303 +/- 0.0027  0.8936 +/- 0.0001   
        2               0.8932 +/- 0.0050  0.8447 +/- 0.0001   
        3               0.8706 +/- 0.0029  0.8096 +/- 0.0004   
        9               0.7772 +/- 0.0028  0.6914 +/- 0.0012   
        19              0.7440 +/- 0.0077  0.5790 +/- 0.0012   

                           micro accuracy    micro precision  \
dataset negative_ratio                                         
ml-1m   1               0.7692 +/- 0.0034  0.7164 +/- 0.0070   
        2               0.7671 +/- 0.0043  0.6125 +/- 0.0082   
        3               0.7892 +/- 0.0023  0.5578 +/- 0.0043   
        9               0.8564 +/- 0.0050  0.3693 +/- 0.0085   
        19              0.8928 +/- 0.0099  0.2446 +/- 0.0150   
steam   1               0.8865 +/- 0.0006  0.8603 +/- 0.0029   
        2               0.8903 +/- 0.0011  0.8033 +/- 0.0053   
        3               0.8977 +/- 0.0007  0.7585 +/- 0.0032   
        9               0.9329 +/- 0.0004  0.6344 +/- 0.0023   
        19              0.9477 +/- 0.0016  0.4855 +/- 0.0098   

                             micro recall           micro F1  
dataset negative_ratio                                        
ml-1m   1               0.8914 +/- 0.0092  0.7943 +/- 0.0008  
        2               0.8215 +/- 0.0126  0.7017 +/- 0.0009  
        3               0.7577 +/- 0.0055  0.6425 +/- 0.0011  
        9               0.6143 +/- 0.0151  0.4611 +/- 0.0030  
        19              0.5418 +/- 0.0318  0.3362 +/- 0.0085  
steam   1               0.9230 +/- 0.0030  0.8905 +/- 0.0002  
        2               0.8886 +/- 0.0052  0.8438 +/- 0.0006  
        3               0.8670 +/- 0.0029  0.8091 +/- 0.0007  
        9               0.7772 +/- 0.0028  0.6986 +/- 0.0009  
        19              0.7440 +/- 0.0077  0.5875 +/- 0.0050

In [4]:
main_metric_comparison = test_summary["macro F1"].unstack("dataset")
main_metric_comparison

dataset,ml-1m,steam
negative_ratio,,
1,0.7941 +/- 0.0006,0.8936 +/- 0.0001
2,0.7014 +/- 0.0005,0.8447 +/- 0.0001
3,0.6408 +/- 0.0007,0.8096 +/- 0.0004
9,0.4570 +/- 0.0024,0.6914 +/- 0.0012
19,0.3282 +/- 0.0049,0.5790 +/- 0.0012


In [5]:
runs[
    [
        "dataset",
        "negative_ratio",
        "seed",
        "test_macro_f1",
        "test_micro_f1",
        "test_macro_accuracy",
        "test_micro_accuracy",
        "threshold",
        "run_dir",
    ]
]

,dataset,negative_ratio,seed,test_macro_f1,test_micro_f1,test_macro_accuracy,test_micro_accuracy,threshold,run_dir
0,ml-1m,1,0,0.794669,0.795394,0.773062,0.773609,143.0,20260608T153011Z_ml-1m_cap20_m1_seed0_populari...
1,ml-1m,1,1,0.793825,0.794106,0.770142,0.770537,139.0,20260608T153040Z_ml-1m_cap20_m1_seed1_populari...
2,ml-1m,1,2,0.794601,0.794676,0.768790,0.768974,129.0,20260608T153108Z_ml-1m_cap20_m1_seed2_populari...
3,ml-1m,1,3,0.793792,0.794073,0.767410,0.768238,130.0,20260608T153137Z_ml-1m_cap20_m1_seed3_populari...
4,ml-1m,1,4,0.793405,0.793243,0.763777,0.764412,120.0,20260608T153206Z_ml-1m_cap20_m1_seed4_populari...
5,ml-1m,2,0,0.701248,0.700955,0.763987,0.764088,191.0,20260608T153234Z_ml-1m_cap20_m2_seed0_populari...
6,ml-1m,2,1,0.701279,0.702901,0.774214,0.774379,214.0,20260608T153304Z_ml-1m_cap20_m2_seed1_populari...
7,ml-1m,2,2,0.702170,0.701901,0.763968,0.764135,188.0,20260608T153333Z_ml-1m_cap20_m2_seed2_populari...
8,ml-1m,2,3,0.700844,0.700733,0.765301,0.765443,198.0,20260608T153402Z_ml-1m_cap20_m2_seed3_populari...
9,ml-1m,2,4,0.701423,0.701791,0.767547,0.767593,196.0,20260608T153431Z_ml-1m_cap20_m2_seed4_populari...
